# [SQL 재현] 2023년 의료기관별 시군구별 진료비 분석

## 단계: 03. 검정·모델링용 데이터 가공 — SQL 재현
- 목표: PY_03·PY_04에서 검정과 모델링 전에 pandas로 만든 변수(광역시·강원더미·보험자부담률·순위·Q3 기준 고진료비)를 SQL로 다시 만들고, 결과가 같은지 대조한다.
- 환경: Jupyter Notebook + sqlite3 (sql_practice.db의 hira_eda 테이블 사용)
- 대조 기준: PY_03_Statistical_Testing.ipynb, PY_04_Modeling.ipynb 실행 결과
- 범위 밖: 정규성·Mann-Whitney·Kruskal-Wallis·Dunn 검정, 회귀 적합 (Python 분석 영역). 단, 마지막에 SQL로 만든 테이블을 Python 모델에 넣어 결과가 같은지 확인한다.

### 3.1 환경 설정
#### 3.1-1 DB 연결 및 저장된 테이블 확인

In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(r'C:\data\sql_practice.db')
pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", conn)

,name
0,hira
1,hira_eda


#### 3.1-2 SQLite 버전·수학 함수 사용 가능 여부 확인
- 윈도우 함수(3.3)는 SQLite 3.25 이상에서만 동작하므로 버전을 확인한다
- SQLite는 설치 방식에 따라 제곱근(`sqrt`)·자연로그(`ln`) 함수가 없을 수 있다 → 함수별로 따로 확인
- `try / except`: 에러가 나도 셀이 멈추지 않고 "사용 불가"로 출력한 뒤 다음 함수로 넘어간다

In [3]:
print(pd.read_sql("SELECT sqlite_version() AS 버전", conn))

for func in ['sqrt(16)', 'ln(10)']:
    try:
        r = pd.read_sql(f"SELECT {func} AS 값", conn)
        print(func, '→ 사용 가능:', r.iloc[0, 0])
    except Exception as e:
        print(func, '→ 사용 불가:', e)

       버전
0  3.51.0
sqrt(16) → 사용 불가: Execution failed on sql 'SELECT sqrt(16) AS 값': no such function: sqrt
ln(10) → 사용 불가: Execution failed on sql 'SELECT ln(10) AS 값': no such function: ln


### 3.2 그룹 변수·비율 변수 생성
#### 3.2-1 모델링용 테이블 만들기 (광역시·강원더미·보험자부담률)
- PY_03 광역시 변수, PY_04 강원더미, PY_03 3.6 보험자부담률 대응 (SAS `IF 시도 IN (...) THEN 광역시 = 1` 대응)
- `CASE WHEN 조건 THEN 1 ELSE 0 END`: 조건을 만족하면 1, 아니면 0인 열을 만든다
- 테이블이 단계별로 쌓이는 구조: hira(원자료) → hira_eda(파생변수) → hira_model(그룹·비율 변수)